In [ ]:
# name: ogn_tcp
# date: 23/07/2026
# author: Toby Alexander

# description: 
# A simple TCP client for connecting to OGN servers. 
# Includes further parsing and filtering of the data stream to examine spatial distribution of messages and identify unique flights.

# required modules:
#   - Python 3.6+ (created with 3.10.19)
#   - cartopy
#   - matplotlib
#   - numpy
#   - ogn.parser (https://github.com/glidernet/python-ogn-client)
#   - pandas
#   - socket

In [ ]:
import cartopy.crs as ccrs      # map plotting library for geospatial data
import cartopy.feature as cfeature
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
from ogn.parser import parse    # for parsing the raw messages from the OGN TCP stream
import pandas as pd
import socket
import threading
import time

# 1. TCP Connection

In [ ]:
# Make a TCP connection to the APRS server (docs: http://www.aprs-is.net/Connecting.aspx)
# Stream data for a specified time, then close the connection and print the collected lines

#------- Functions ---------

def send_heartbeats(sock, interval):
    """Periodically send an APRS-IS comment line to keep the connection alive."""
    while not stop_event.wait(interval):
        try:
            sock.sendall(b"#keepalive\r\n")
            # print(f"DEBUG: [{time.strftime('%H:%M:%S')}] sent heartbeat")
        except OSError:
            # print(f"DEBUG: [{time.strftime('%H:%M:%S')}] heartbeat send failed: {OSError}")
            break  # socket already closed

#------- Code Start ------- 

# Specify the host and port for the APRS server
host = 'aprs.glidernet.org'
port = 14580

# Create the login string for the APRS server connection.
# Specify a unique username, no more than 9 characters, e.g. user mycall
# Usually a password is required, but -1 can be used for read-only access.
# Specify a filter to limit the data stream to a specific geographic area: r/lat/long/dist (latitude, longitude, radius in km).
# Specify Pass traffic with p/FLR to receive only FLARM messages. Without this, we found that some ADS-B messages are also received.
login_str = "user example1 -1 filter r/53.6/-4.2/800 p/FLR\r\n"

lines = []

start_time = time.time()
stream_time_hr = 1/60  # hours to stream data for
end_time = start_time + 3600 * stream_time_hr  # Run for specified time

keepalive_interval_sec = 60  # send a heartbeat every 60s of idle time

stop_event = threading.Event()

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect((host, port))

    stream = s.makefile('r', encoding='utf-8', errors='replace')

    s.sendall(login_str.encode('utf-8'))

    # Start background thread to send periodic keepalives
    heartbeat_thread = threading.Thread(
        target=send_heartbeats, args=(s, keepalive_interval_sec), daemon=True
    )
    heartbeat_thread.start()

    try:
        for line in stream:
            lines.append(line.strip())
            if time.time() > end_time:
                break  # Exit loop
    finally:
        stop_event.set()  # tell heartbeat thread to stop
        s.shutdown(socket.SHUT_RDWR)
        s.close()

# Socket is now closed, examine lines freely
print(f"Collected {len(lines)} lines")

In [ ]:
# Display first few lines.
for line in lines[0:10]:
    print(line)

In [ ]:
# Save lines to a text file for later analysis if required. The filename includes the date and time range of the data stream.
filename = 'aprs_data_20260322_T1534-1604'
with open(f"{filename}.txt", 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

# 2. Data Parsing

FLARM data comes as a raw message, which can be parsed (using ogn.parser.parse) into a dictionary containing the various contents of the message such as timestamps, lat/lon, flight IDs, etc.

In [ ]:
# Define function to parse the collected APRS lines into a pandas DataFrame using the ogn.parser.parse function.

def parse_aprs_lines(lines):
    """
    Parse a list of APRS lines into a list of dictionaries using the ogn.parser.parse function.
    Handles unparseable lines by skipping them.
    Returns a pandas DataFrame.

    lines: list of strings, each string is a line from the APRS data stream.
    """

    records = []
    for line in lines:
        line = line.strip()     # remove leading and trailing whitespace, including \r\n
        if not line or line.startswith('#'):  # skip empty lines and comments
            continue
        try:
            parsed = parse(line)    # Calls the ogn.parser.parse function for each line.
            if parsed:      # parse function returns None for unparseable lines, so check before appending
                records.append(parsed)
        except Exception:
            continue  # skip unparseable lines
    return pd.DataFrame(records)

In [ ]:
# Read lines back in from file and parse into DataFrame using ogn.parser.parse function.
root = ""
filename = ""
with open(root+filename, 'r') as f:
    df = parse_aprs_lines(f)

In [ ]:
# Alternatively, if you have the lines already in memory, you can parse them directly:
df = parse_aprs_lines(lines)  # parse the collected lines into a DataFrame

In [ ]:
# Display summary info about the DataFrame, including column names, data types
df.info()

# 3. Filtering

Other aircraft use the FLARM system, so we need to filter for gliders only. Also seem to have an issue with receiving messages which are outside the specified APRS filter, so we can filter those out too.

In [ ]:
# As specified in the OGN documentation (http://wiki.glidernet.org/wiki:ogn-flavoured-aprs), aircraft_type = 1 corresponds to gliders, so filter the DataFrame to only include those records for now.
gliders_df = df[df['aircraft_type'] == 1].copy()
gliders_df['timestamp_unix'] = gliders_df['timestamp'].astype('int64') / 10**9  # Create a new column for the timestamp in unix time (seconds since epoch) for easier filtering and plotting.

print(f"Total records: {len(df)}, Glider records: {len(gliders_df)} ({len(gliders_df)/len(df)*100:.2f}%)")

In [ ]:
# find measurements outside the UK bounding box (49.5N to 61N, -8.5E to 2E)
# For some reason many messages outside of the filtered range are still being received from EU and US, so check how many and which they are.

# Define a square bounding box for the UK (approximate)
uk_bb_lat = [49.5, 61]
uk_bb_lon = [-8.5, 2]

mask_outside = (
    (gliders_df['latitude'] < uk_bb_lat[0]) | (gliders_df['latitude'] > uk_bb_lat[1]) |
    (gliders_df['longitude'] < uk_bb_lon[0]) | (gliders_df['longitude'] > uk_bb_lon[1])
)
print(f"Messages outside bounding box: {mask_outside.sum()} ({mask_outside.sum() / len(gliders_df) * 100:.2f}%)")
print(gliders_df[mask_outside][['latitude', 'longitude', 'timestamp', 'name']].head(5))

In [ ]:
# Similarly some messages outside of the time range are also being received, so check how many and which they are.
mask_time = (gliders_df['timestamp_unix'] >= start_time) & (gliders_df['timestamp_unix'] <= end_time)
print(f"Points outside time range: {(~mask_time).sum()} ({(~mask_time).sum() / len(gliders_df) * 100:.2f}%)")

In [ ]:
# Filter for messages within a bounding box around the UK and within the specified time range, and print how many points remain.
# Create a new dataframe with only UK glider messages within the correct time range.
mask_inside = (
    (gliders_df['latitude'] >= uk_bb_lat[0]) & (gliders_df['latitude'] <= uk_bb_lat[1]) &
    (gliders_df['longitude'] >= uk_bb_lon[0]) & (gliders_df['longitude'] <= uk_bb_lon[1]) &
    (gliders_df['timestamp'].astype('int64') / 10**9 >= start_time) & (gliders_df['timestamp'].astype('int64') / 10**9 <= end_time)
)

gliders_uk_df = gliders_df[mask_inside].copy()
print(f"Points inside bounding box and time range: {len(gliders_uk_df)}")

In [ ]:
# Visualise the distribution of glider messages over the UK for the filtered data, coloured by time of day.

def plot_messages(df, lat_col='latitude', lon_col='longitude'):

    """
    Plot the spatial distribution of glider messages on a map, coloured by time of day.
     - df: DataFrame containing the glider messages, with columns for latitude, longitude, and timestamp.
     - lat_col, lon_col: names of the columns in df that contain latitude and longitude values, respectively.
     - The function creates a map using Cartopy, plots the glider message locations as a scatter plot coloured by time of day, and displays the map with appropriate features and labels.
     - The timestamp is converted to hours of the day (0-24) for colouring the points, and a colorbar is added to indicate the time scale.
    """

    plt.rcParams.update({
        'font.size': 12,
        'axes.titlesize': 12,
        'axes.labelsize': 10,
        'xtick.labelsize': 10,
        'ytick.labelsize': 12,
        'legend.fontsize': 12,
    })

    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
    
    ax.add_feature(cfeature.LAND, facecolor='lightgrey')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle='--')
    ax.add_feature(cfeature.RIVERS, linewidth=0.3)
    
    margin = 0.2

    all_lons = list(df[lon_col])
    all_lats = list(df[lat_col])

    ax.set_extent([min(all_lons) - margin, max(all_lons) + margin,
                min(all_lats) - margin, max(all_lats) + margin])
    
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False

    t_hours = df['timestamp'].astype('int64') // 10**9 % (3600 * 24) / 3600
    vmin, vmax = t_hours.min(), t_hours.max()

    scatter = ax.scatter(df[lon_col], df[lat_col],
                         c=t_hours, cmap='viridis', s=20, alpha=0.7,
                         vmin=vmin, vmax=vmax,
                         transform=ccrs.PlateCarree(),
                         label='Glider messages')

    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label('Time (hr of day)')
    ax.set_title('Spatial distribution of glider messages')
    ax.legend(loc='upper left')
    fig.tight_layout()
    plt.show()

plot_messages(gliders_uk_df, lat_col='latitude', lon_col='longitude')

In [ ]:
# Save the filtered DataFrame to a Parquet file for later analysis. Parquet is more efficient than CSV, and can be read directly in with pandas for analysis.

gliders_uk_df.to_parquet('aprs_data_20260322_T1534-1604_UKfilter.parquet')  # modify filename as needed for your data stream

In [ ]:
# Read back in the Parquet file if required. Otherwise, the gliders_uk_df DataFrame is already in memory and can be used for further analysis.
gliders_uk_df = pd.read_parquet('aprs_data_20260322_T1534-1604_UKfilter.parquet')   # modify filepath as needed.

# 4. Grouping of messages

In [ ]:
# Now with the messages filtered, we can group them into individual flight tracks.
# Group the glider messages by aircraft (using the name identifier) and store in a dictionary 'unq_f', where the keys are the aircraft names and the values are DataFrames containing the messages for each aircraft.

unq_f = {}      # unique flights
grouped = gliders_uk_df.groupby('name')      # Group rows in the DataFrame by the 'name' column, which identifies individual aircraft

for icao, group in grouped:
    unq_f[icao] = group

print(f"num of unique aircraft: {len(unq_f.keys())}")
unq_f.keys()
nmes = np.empty(len(unq_f.keys()))
keys = np.empty(len(unq_f.keys()), dtype='U32')  # Use string dtype for keys

i = 0
for key in unq_f.keys():
    nmes[i], keys[i] = len(unq_f[key]), key     # nmes: number of messages for this aircraft; key: name of the aircraft
    i += 1

# Sort keys and nmes together in descending order of nmes
sorted_indices = np.argsort(nmes)[::-1]     # Get indices that would sort
nmes = nmes[sorted_indices]                 # Sort nmes
keys = keys[sorted_indices]                 # Sort keys using the same indices

# Looking for around 1000 messages or more for a flight track to be of sufficient length to observe thermal soaring.
# Often partial tracks are picked up since we taken a snippet in time, so may catch only the beginning or end of a flight, and some gliders may fly beyond receiver range.
print(f"number of flights with > 1000 messages: {np.sum(nmes > 1000)}")

In [ ]:
# Select a specific flight by setting keyidx. 
# The flights are sorted by message length, so keyidx = 0 corresponds to the most active glider, keyidx = 1 the second most active, and so on.
keyidx = 0
gkey = keys[keyidx]
print(f"key: {gkey}, number of messages: {nmes[keyidx]}")
# print(unq_f[gkey].head())

g1df = unq_f[gkey]
g1df = g1df.sort_values('timestamp')
g1df = g1df.reset_index(drop=True)

In [ ]:
# Examine the flight path by looking at altitude vs time, and longitude vs latitude. 
# A sawtooth pattern in altitude is a strong indicator of thermal soaring, as the glider climbs in a thermal and then glides down to the next thermal, and so on.

fig = plt.figure(figsize=(4, 3))
# plt.plot(g1df.timestamp[g1df.receiver_name==recvname], g1df.altitude[g1df.receiver_name==recvname], marker='o', linestyle='', markersize=3, label = recvname)
plt.plot(g1df.timestamp, g1df.altitude, marker='o', linestyle='', markersize=3, label = gkey)

plt.legend()
plt.xlabel('Time')  
plt.ylabel('Altitude (m)')
plt.title(f"Altitude vs Time for {gkey}")
plt.show()

fig = plt.figure(figsize=(6, 4))
plt.plot(g1df.longitude[40:-1], g1df.latitude[40:-1], marker='o', linestyle='-', markersize=2, label = gkey, alpha=0.7)
plt.legend()
plt.xlabel('longitude')
plt.ylabel('latitude')
plt.title(f"Flight path for {gkey}")

In [ ]:
# Save out the flight track for this glider to a Parquet file for later analysis.

filename = f"flarm_aprs_{gkey}.parquet"

g1df.to_parquet(filename, index=False)